# Data Analysis

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import date
from pathlib import Path

import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = [10, 7]
plt.style.use("seaborn-v0_8")

import seaborn as sns

sns.set(style="darkgrid")

import ipywidgets as widgets
import pandas as pd

## Constants

In [ ]:
PROJECT_ROOT = Path("__file__").resolve().parents[1]

DATA_DPATH = PROJECT_ROOT / "data"
assert DATA_DPATH.exists()

## Data Loading 

In [ ]:
train_fpath = DATA_DPATH / "src_data" / "train.csv"
train_df = pd.read_csv(train_fpath)
train_df.shape

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df["date"] = pd.to_datetime(train_df["date"])

In [ ]:
test_fpath = DATA_DPATH / "src_data" / "test.csv"
test_df = pd.read_csv(test_fpath, parse_dates=["date"])
test_df.shape

In [ ]:
test_df.head()

## Duplicates

In [ ]:
train_df[train_df.duplicated()]

In [ ]:
test_df[test_df.duplicated()]

## Missing Values

In [ ]:
train_df.isna().sum()

In [ ]:
train_df.isna().sum() / len(train_df)

There are about 4% of missing values in target column

In [ ]:
test_df.isna().sum()

In [ ]:
test_df.isna().sum() / len(test_df)

## Date Limits

In [ ]:
train_df["date"].describe()

In [ ]:
test_df["date"].describe()

## Features

In [ ]:
train_df["country"].value_counts(dropna=False)

In [ ]:
test_df["country"].value_counts(dropna=False)

In [ ]:
train_df["store"].value_counts(dropna=False)

In [ ]:
test_df["store"].value_counts(dropna=False)

In [ ]:
train_df["product"].value_counts(dropna=False)

In [ ]:
test_df["product"].value_counts(dropna=False)

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 4))

plot_data = train_df["country"].value_counts(dropna=False)
plot_data = plot_data.to_frame(name="cnt").reset_index(names="country")
plot_data["prc"] = plot_data["cnt"] / plot_data["cnt"].sum()
plot_data["labels"] = (plot_data["prc"] * 100).round(1).astype(str) + "%"
plot_data = plot_data.sort_values("country")

sns.barplot(plot_data, x="country", y="cnt", hue="country", ax=axs[0])

for c, label in zip(axs[0].containers, plot_data["labels"], strict=True):
    axs[0].bar_label(c, [label])

axs[0].set_xlabel("Country Name")
axs[0].set_ylabel("Number of Orders")
axs[0].set_title("Country Orders Distribution")


sns.countplot(train_df, x="country", hue="store", ax=axs[1])

axs[1].set_xlabel("Country Name")
axs[1].set_ylabel("Number of orders")
axs[1].set_title("Contry Orders Distribution by Stores")

plt.tight_layout()
plt.show()

In [ ]:
@widgets.interact(
    country=widgets.Dropdown(options=train_df["country"].unique()),
    store=widgets.Dropdown(options=train_df["store"].unique()),
    products=widgets.Dropdown(options=train_df["product"].unique()),
    start_date=widgets.DatePicker(value=train_df["date"].min()),
)
def show_country_store_product_data(country: str, store: str, products: str, start_date: date):
    start_date = pd.Timestamp(start_date)
    plot_df = train_df[
        (train_df["country"] == country)
        & (train_df["store"] == store)
        & (train_df["product"] == products)
        & (train_df["date"].dt.date >= start_date.date())
    ]

    fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(15, 4))

    axs.plot(plot_df["date"], plot_df["num_sold"])
    axs.set_xlabel("Date")
    axs.set_ylabel("Number of sold stickers")

    plt.show()

## Target

In [ ]:
train_df["num_sold"].describe()

In [ ]:
sns.catplot(
    data=train_df,
    x="country",
    y="num_sold",
    kind="boxen",
    aspect=1.5,
    palette="Set2",
)

plt.xlabel("Country Name")
plt.ylabel("Number of Sold Stickers")
plt.title("Distribution of Sold Stickers by Country")

plt.show()

In [ ]:
sns.catplot(
    data=train_df,
    x="store",
    y="num_sold",
    kind="boxen",
    aspect=1.5,
    palette="Set2",
)

plt.xlabel("Store Name")
plt.ylabel("Number of Sold Stickers")
plt.title("Distribution of Sold Stickers by Store")

plt.show()

In [ ]:
sns.catplot(
    data=train_df,
    x="product",
    y="num_sold",
    kind="boxen",
    aspect=1.5,
    palette="Set2",
)

plt.xlabel("Product Name")
plt.ylabel("Number of Sold Stickers")
plt.title("Distribution of Sold Stickers by Product")

plt.show()

In [ ]:
@widgets.interact(
    country=widgets.Dropdown(options=train_df["country"].unique()),
    store=widgets.Dropdown(options=train_df["store"].unique()),
    products=widgets.Dropdown(options=train_df["product"].unique()),
)
def show_target_statistics(country: str, store: str, products: str):
    plot_df = train_df[
        (train_df["country"] == country)
        & (train_df["store"] == store)
        & (train_df["product"] == products)
    ]

    na_val = plot_df["num_sold"].isna().sum()
    print(f"Missing values: {na_val} ({na_val / len(plot_df) * 100:.1f}%)")

    if na_val != len(plot_df):
        _fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 4))

        sns.histplot(
            plot_df,
            x="num_sold",
            kde=True,
            bins=30,
            log_scale=False,
            ax=axs[0],
        )
        axs[0].set_xlabel("Number of Sold Stickers")
        axs[0].set_ylabel("Sample Number")
        axs[0].set_title("Number of Sold Stickers Distribution")

        sns.boxplot(plot_df["num_sold"], ax=axs[1])
        axs[1].set_ylabel("Sample Number")
        axs[1].set_title("Sold Sticker Boxplot")

        plt.tight_layout()
        plt.show()

In [ ]:
train_df["segment"] = train_df["country"] + "_" + train_df["store"] + "_" + train_df["product"]
train_df["segment"].nunique()

In [ ]:
test_df["segment"] = test_df["country"] + "_" + test_df["store"] + "_" + test_df["product"]
test_df["segment"].nunique()

In [ ]:
train_grouped = train_df.groupby(["segment"], as_index=False)["num_sold"].mean()

train_empty_segments = train_grouped[train_grouped["num_sold"].isna()]
train_empty_segments["country"] = train_empty_segments["segment"].apply(lambda x: x.split("_")[0])
train_empty_segments["store"] = train_empty_segments["segment"].apply(lambda x: x.split("_")[1])
train_empty_segments["product"] = train_empty_segments["segment"].apply(lambda x: x.split("_")[2])

train_empty_segments

In [ ]:
@widgets.interact(
    country=widgets.Dropdown(options=train_df["country"].unique()),
    store=widgets.Dropdown(options=train_empty_segments["store"].unique()),
    products=widgets.Dropdown(options=train_empty_segments["product"].unique()),
)
def show_empty_similar_data(country: str, store: str, products: str):
    plot_df = train_df[
        (train_df["country"] == country)
        & (train_df["store"] == store)
        & (train_df["product"] == products)
    ]

    fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(15, 4))

    axs.plot(plot_df["date"], plot_df["num_sold"])
    axs.set_xlabel("Date")
    axs.set_ylabel("Number of sold stickers")

    plt.show()